In [ ]:
# Data Processing Libraries
import pandas as pd
import numpy as np

# SQL Integration
import sqlalchemy
import sqlite3
import duckdb

# Visualization Libraries
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
# Read the Excel file
df = pd.read_excel('Cleaned_Data.xlsx')

# Display the first few rows to verify it loaded correctly
df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target,Flight Risk
0,1,8,5,2,1,1,1,13,10,6,...,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout,1
1,1,6,1,11,1,1,1,1,3,4,...,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate,0
2,1,1,5,5,1,1,1,22,27,10,...,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout,1
3,1,8,2,15,1,1,1,23,27,6,...,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate,0
4,2,12,1,3,0,1,1,22,28,10,...,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate,0


In [ ]:
# Check the dimensions of the dataset
print(f"Dataset Shape: {df.shape}")

# View data types and identify any missing values
print("\n--- Data Info ---")
df.info()

# Get statistical summaries for numerical columns
print("\n--- Statistical Summary ---")
df.describe()

# Check for any missing values across the dataset
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

Dataset Shape: (4424, 36)

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 36 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Nacionality                                     4424 non-null   int64  
 7   Mother's qualification                          4424 non-null   int64  
 8   Father's qualification                          4424 non-null   int64  
 

In [ ]:
# Create a connection to an in-memory DuckDB database
con = duckdb.connect(database=':memory:')

# Register the pandas DataFrame as a virtual table
con.register('student_data', df)

# Verify by running a simple SQL query
query_result = con.execute("SELECT * FROM student_data LIMIT 5").df()
query_result

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target,Flight Risk
0,1,8,5,2,1,1,1,13,10,6,...,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout,1
1,1,6,1,11,1,1,1,1,3,4,...,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate,0
2,1,1,5,5,1,1,1,22,27,10,...,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout,1
3,1,8,2,15,1,1,1,23,27,6,...,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate,0
4,2,12,1,3,0,1,1,22,28,10,...,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate,0


# **Primary Demographic**

**Nationality **

In [ ]:

nationality_risk = con.execute("""
    SELECT
        Nacionality,
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY Nacionality
    ORDER BY total_students DESC
""").df()

# Display the results
nationality_risk

,Nacionality,total_students,dropout_rate_pct,avg_sem1_grade
0,1,4314,32.20,10.64
1,14,38,36.84,10.19
2,12,14,7.14,10.79
3,9,13,30.77,10.85
4,3,13,30.77,9.92
5,10,5,20.00,12.30
6,18,3,33.33,12.53
7,16,3,66.67,7.83
8,4,3,0.00,13.66
9,15,2,0.00,14.06


In [ ]:
#bar chart for Nationality Risk
fig = px.bar(
    nationality_risk,
    x='Nacionality',
    y='dropout_rate_pct',
    title='Dropout Rate by Nationality ID',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'Nacionality': 'Nationality ID'},
    color='dropout_rate_pct',
    color_continuous_scale='Reds',
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

# formatting
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(
    xaxis_type='category', # Treat IDs as labels, not numbers
    yaxis_title="Dropout Rate (%)",
    height=500
)

fig.show()

1 - Portuguese
2 - German
6 - Spanish
11 - Italian
13 - Dutch
14 - English
17 - Lithuanian
21 - Angolan
22 - Cape Verdean
24 - Guinean
25 - Mozambican
26 - Santomean
32 - Turkish
41 - Brazilian
62 - Romanian
100 - Moldova (Republic of)
101 - Mexican
103 - Ukrainian
105 - Russian
108 - Cuban
109 - Colombian

**Gender**

In [ ]:
# analyze dropout risk by Gender
gender_risk = con.execute("""
    SELECT
        Gender,
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY Gender
    ORDER BY Gender ASC
""").df()

# Mapping the numeric codes to labels for better readability
gender_risk['Gender_Label'] = gender_risk['Gender'].map({1: 'Male', 0: 'Female'})

gender_risk

,Gender,total_students,dropout_rate_pct,avg_sem1_grade,Gender_Label
0,0,2868,25.10,11.32,Female
1,1,1556,45.05,9.40,Male


In [ ]:
# Create a bar chart for Gender Risk
fig = px.bar(
    gender_risk,
    x='Gender_Label',
    y='dropout_rate_pct',
    title='Dropout Rate by Gender',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'Gender_Label': 'Gender'},
    color='Gender_Label',
    color_discrete_map={'Male': 'royalblue', 'Female': 'pink'},
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=400)
fig.show()

**Marital Status**

In [ ]:
# dropout risk by Marital status
marital_risk = con.execute("""
    SELECT
        "Marital status",
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY "Marital status"
    ORDER BY total_students DESC
""").df()

# Mapping common IDs to labels
marital_labels = {1: 'Single', 2: 'Married', 3: 'Widower', 4: 'Divorced', 5: 'Facto Union', 6: 'Legally Separated'}
marital_risk['Marital_Label'] = marital_risk['Marital status'].map(marital_labels)

marital_risk

,Marital status,total_students,dropout_rate_pct,avg_sem1_grade,Marital_Label
0,1,3919,30.21,10.76,Single
1,2,379,47.23,9.84,Married
2,4,91,46.15,9.50,Divorced
3,5,25,44.00,10.88,Facto Union
4,6,6,66.67,5.83,Legally Separated
5,3,4,25.00,5.50,Widower


In [ ]:
# Create a bar chart for Marital Status Risk
fig = px.bar(
    marital_risk,
    x='Marital_Label',
    y='dropout_rate_pct',
    title='Dropout Rate by Marital Status',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'Marital_Label': 'Marital Status'},
    color='dropout_rate_pct',
    color_continuous_scale='Purples',
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(xaxis_title="Marital Status", yaxis_title="Dropout Rate (%)", height=500)
fig.show()

**Gender and Marital Status**

In [ ]:
#interaction of Gender and Marital Status
interaction_data = con.execute("""
    SELECT
        Gender,
        "Marital status",
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct
    FROM student_data
    GROUP BY Gender, "Marital status"
    ORDER BY Gender, "Marital status"
""").df()

# Apply labels for readability
interaction_data['Gender_Label'] = interaction_data['Gender'].map({1: 'Male', 0: 'Female'})
interaction_data['Marital_Label'] = interaction_data['Marital status'].map({
    1: 'Single', 2: 'Married', 3: 'Widower', 4: 'Divorced', 5: 'Facto Union', 6: 'Legally Separated'
})

interaction_data

,Gender,Marital status,total_students,dropout_rate_pct,Gender_Label,Marital_Label
0,0,1,2552,23.16,Female,Single
1,0,2,221,40.27,Female,Married
2,0,3,3,33.33,Female,Widower
3,0,4,69,40.58,Female,Divorced
4,0,5,18,44.44,Female,Facto Union
5,0,6,5,60.00,Female,Legally Separated
6,1,1,1367,43.38,Male,Single
7,1,2,158,56.96,Male,Married
8,1,3,1,0.00,Male,Widower
9,1,4,22,63.64,Male,Divorced


In [ ]:
# Create a pivot table for the heatmap
heatmap_df = interaction_data.pivot(index='Marital_Label', columns='Gender_Label', values='dropout_rate_pct')

# Create the heatmap
fig = px.imshow(
    heatmap_df,
    labels=dict(x="Gender", y="Marital Status", color="Dropout Rate (%)"),
    x=['Female', 'Male'],
    y=heatmap_df.index,
    color_continuous_scale='Reds',
    title='Dropout Risk Heatmap: Gender vs. Marital Status',
    text_auto='.1f' # Shows the percentage directly on the tiles
)

fig.update_layout(height=500)
fig.show()

**Age**

In [ ]:
# SQL query to analyze dropout risk by Age Groups
age_risk = con.execute("""
    SELECT
        CASE
            WHEN "Age at enrollment" <= 21 THEN 'Traditional (18-21)'
            WHEN "Age at enrollment" BETWEEN 22 AND 30 THEN 'Young Adult (22-30)'
            ELSE 'Mature (31+)'
        END AS age_group,
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY age_group
    ORDER BY age_group ASC
""").df()

age_risk

,age_group,total_students,dropout_rate_pct,avg_sem1_grade
0,Mature (31+),662,53.63,9.26
1,Traditional (18-21),2873,22.10,11.28
2,Young Adult (22-30),889,48.48,9.59


In [ ]:
# Create a bar chart for Age Risk
fig = px.bar(
    age_risk,
    x='age_group',
    y='dropout_rate_pct',
    title='Dropout Rate by Age at Enrollment',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'age_group': 'Age Group'},
    color='dropout_rate_pct',
    color_continuous_scale='Oranges',
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=500)
fig.show()

**Age and Attendence**

In [ ]:
# SQL query for the interaction of Age and Attendance Mode
age_attendance_data = con.execute("""
    SELECT
        CASE
            WHEN "Age at enrollment" <= 21 THEN 'Traditional (18-21)'
            WHEN "Age at enrollment" BETWEEN 22 AND 30 THEN 'Young Adult (22-30)'
            ELSE 'Mature (31+)'
        END AS age_group,
        "Daytime/evening attendance",
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY age_group, "Daytime/evening attendance"
    ORDER BY age_group ASC
""").df()

# Map the attendance codes to labels (1 = Daytime, 0 = Evening)
age_attendance_data['Attendance'] = age_attendance_data['Daytime/evening attendance'].map({1: 'Daytime', 0: 'Evening'})

age_attendance_data

,age_group,Daytime/evening attendance,total_students,dropout_rate_pct,avg_sem1_grade,Attendance
0,Mature (31+),1,386,60.88,9.08,Daytime
1,Mature (31+),0,276,43.48,9.50,Evening
2,Traditional (18-21),1,2841,22.10,11.28,Daytime
3,Traditional (18-21),0,32,21.88,11.41,Evening
4,Young Adult (22-30),1,714,49.16,9.53,Daytime
5,Young Adult (22-30),0,175,45.71,9.86,Evening


In [ ]:
# Create a grouped bar chart for Age and Attendance interaction
fig = px.bar(
    age_attendance_data,
    x='age_group',
    y='dropout_rate_pct',
    color='Attendance',
    barmode='group',
    title='Dropout Risk: Age Group vs. Attendance Mode',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'age_group': 'Age Group'},
    color_discrete_map={'Daytime': 'orange', 'Evening': 'navy'},
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=500)
fig.show()

**Special Education**

In [ ]:
# SQL query to analyze dropout risk by Educational Special Needs
special_needs_risk = con.execute("""
    SELECT
        "Educational special needs",
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY "Educational special needs"
    ORDER BY "Educational special needs" ASC
""").df()

# Mapping the numeric codes to labels (1 = Yes, 0 = No)
special_needs_risk['Needs_Label'] = special_needs_risk['Educational special needs'].map({1: 'Yes', 0: 'No'})

special_needs_risk

,Educational special needs,total_students,dropout_rate_pct,avg_sem1_grade,Needs_Label
0,0,4373,32.11,10.65,No
1,1,51,33.33,10.09,Yes


In [ ]:
# Create a bar chart for Educational Special Needs Risk
fig = px.bar(
    special_needs_risk,
    x='Needs_Label',
    y='dropout_rate_pct',
    title='Dropout Rate by Educational Special Needs',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'Needs_Label': 'Special Needs'},
    color='Needs_Label',
    color_discrete_map={'Yes': 'crimson', 'No': 'lightgrey'},
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=450)
fig.show()

In [ ]:
# Create a pie chart for the distribution of Educational Special Needs
fig = px.pie(
    special_needs_risk,
    values='total_students',
    names='Needs_Label',
    title='Percentage of Students with Educational Special Needs',
    color='Needs_Label',
    color_discrete_map={'Yes': 'crimson', 'No': 'lightgrey'},
    hole=0.4 # This makes it a donut chart, which is often easier to read
)

# Add styling to show both labels and percentages
fig.update_traces(textinfo='percent+label')
fig.show()

# **Socio-Economic Factors**

**Debt**

In [ ]:
# analyze dropout risk by Debtor status
debtor_risk = con.execute("""
    SELECT
        Debtor,
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY Debtor
    ORDER BY Debtor ASC
""").df()

# Mapping the numeric codes to labels (1 = Yes, 0 = No)
debtor_risk['Debtor_Label'] = debtor_risk['Debtor'].map({1: 'Yes', 0: 'No'})

debtor_risk

,Debtor,total_students,dropout_rate_pct,avg_sem1_grade,Debtor_Label
0,0,3921,28.28,10.82,No
1,1,503,62.03,9.23,Yes


In [ ]:
# Create a bar chart for Debtor Risk
fig = px.bar(
    debtor_risk,
    x='Debtor_Label',
    y='dropout_rate_pct',
    title='Dropout Rate by Debtor Status',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'Debtor_Label': 'Has Debt'},
    color='Debtor_Label',
    color_discrete_map={'Yes': 'darkred', 'No': 'seagreen'},
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=450)
fig.show()

**Scholarship **

In [ ]:
#nalyze dropout risk by Scholarship holder status
scholarship_risk = con.execute("""
    SELECT
        "Scholarship holder",
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY "Scholarship holder"
    ORDER BY "Scholarship holder" ASC
""").df()

# Mapping the numeric codes to labels (1 = Yes, 0 = No)
scholarship_risk['Scholarship_Label'] = scholarship_risk['Scholarship holder'].map({1: 'Yes', 0: 'No'})

scholarship_risk

,Scholarship holder,total_students,dropout_rate_pct,avg_sem1_grade,Scholarship_Label
0,0,3325,38.71,10.17,No
1,1,1099,12.19,12.06,Yes


In [ ]:
# Create a bar chart for Scholarship Risk
fig = px.bar(
    scholarship_risk,
    x='Scholarship_Label',
    y='dropout_rate_pct',
    title='Dropout Rate by Scholarship Status',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'Scholarship_Label': 'Scholarship Holder'},
    color='Scholarship_Label',
    color_discrete_map={'Yes': 'gold', 'No': 'lightslategrey'},
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=450)
fig.show()

**Debt and Scholarship**

In [ ]:
# interaction of Scholarship and Debtor status
financial_interaction = con.execute("""
    SELECT
        "Scholarship holder",
        "Debtor",
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY "Scholarship holder", "Debtor"
    ORDER BY "Scholarship holder", "Debtor"
""").df()

# Apply labels for readability
financial_interaction['Scholarship_Label'] = financial_interaction['Scholarship holder'].map({1: 'Scholarship', 0: 'No Scholarship'})
financial_interaction['Debtor_Label'] = financial_interaction['Debtor'].map({1: 'Has Debt', 0: 'No Debt'})

financial_interaction

,Scholarship holder,Debtor,total_students,dropout_rate_pct,avg_sem1_grade,Scholarship_Label,Debtor_Label
0,0,0,2906,34.62,10.35,No Scholarship,No Debt
1,0,1,419,67.06,8.91,No Scholarship,Has Debt
2,1,0,1015,10.15,12.16,Scholarship,No Debt
3,1,1,84,36.90,10.85,Scholarship,Has Debt


In [ ]:
# Create a grouped bar chart for Financial Interaction
fig = px.bar(
    financial_interaction,
    x='Scholarship_Label',
    y='dropout_rate_pct',
    color='Debtor_Label',
    barmode='group',
    title='Dropout Risk: Scholarship Status vs. Debtor Status',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'Scholarship_Label': 'Scholarship Status'},
    color_discrete_map={'Has Debt': 'darkred', 'No Debt': 'seagreen'},
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=500)
fig.show()

**Tuition Fees Up-To-Date**

In [ ]:
# analyze dropout risk by Tuition fees status
tuition_risk = con.execute("""
    SELECT
        "Tuition fees up to date",
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY "Tuition fees up to date"
    ORDER BY "Tuition fees up to date" ASC
""").df()

# Mapping the numeric codes to labels (1 = Up to Date, 0 = Overdue)
tuition_risk['Tuition_Label'] = tuition_risk['Tuition fees up to date'].map({1: 'Up to Date', 0: 'Overdue'})

tuition_risk

,Tuition fees up to date,total_students,dropout_rate_pct,avg_sem1_grade,Tuition_Label
0,0,528,86.55,7.35,Overdue
1,1,3896,24.74,11.09,Up to Date


In [ ]:
# Create a bar chart for Tuition Fees Risk
fig = px.bar(
    tuition_risk,
    x='Tuition_Label',
    y='dropout_rate_pct',
    title='Dropout Rate by Tuition Fees Status',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'Tuition_Label': 'Tuition Fees Status'},
    color='Tuition_Label',
    color_discrete_map={'Up to Date': 'seagreen', 'Overdue': 'darkred'},
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=450)
fig.show()

**Displaced Status**

In [ ]:
# analyze dropout risk by Displaced status
displaced_risk = con.execute("""
    SELECT
        Displaced,
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY Displaced
    ORDER BY Displaced ASC
""").df()

# Mapping the numeric codes to labels (1 = Yes, 0 = No)
displaced_risk['Displaced_Label'] = displaced_risk['Displaced'].map({1: 'Yes', 0: 'No'})

displaced_risk

,Displaced,total_students,dropout_rate_pct,avg_sem1_grade,Displaced_Label
0,0,1998,37.64,10.29,No
1,1,2426,27.58,10.93,Yes


In [ ]:
# Create a bar chart for Displaced Status Risk
fig = px.bar(
    displaced_risk,
    x='Displaced_Label',
    y='dropout_rate_pct',
    title='Dropout Rate by Displaced Status',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'Displaced_Label': 'Is Displaced'},
    color='Displaced_Label',
    color_discrete_map={'Yes': 'dodgerblue', 'No': 'lightgray'},
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=450)
fig.show()

**Displaced × International**

In [ ]:
# SQL query for the interaction of Displaced and International status
displaced_int_interaction = con.execute("""
    SELECT
        Displaced,
        International,
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY Displaced, International
    ORDER BY Displaced, International
""").df()

# Apply labels for readability
displaced_int_interaction['Displaced_Label'] = displaced_int_interaction['Displaced'].map({1: 'Displaced', 0: 'Not Displaced'})
displaced_int_interaction['International_Label'] = displaced_int_interaction['International'].map({1: 'International', 0: 'Domestic'})

displaced_int_interaction

,Displaced,International,total_students,dropout_rate_pct,avg_sem1_grade,Displaced_Label,International_Label
0,0,0,1945,37.69,10.30,Not Displaced,Domestic
1,0,1,53,35.85,10.10,Not Displaced,International
2,1,0,2369,27.69,10.91,Displaced,Domestic
3,1,1,57,22.81,11.52,Displaced,International


In [ ]:
# Create a grouped bar chart for Displaced and International interaction
fig = px.bar(
    displaced_int_interaction,
    x='Displaced_Label',
    y='dropout_rate_pct',
    color='International_Label',
    barmode='group',
    title='Dropout Risk: Displaced vs. International Status',
    labels={'dropout_rate_pct': 'Dropout Rate (%)', 'Displaced_Label': 'Displaced Status'},
    color_discrete_map={'International': 'gold', 'Domestic': 'dodgerblue'},
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100], height=500)
fig.show()

# **Family Factors**

**Mother's Qualifications**

1 - Secondary Education - 12th Year of Schooling or Eq. 2 - Higher Education - Bachelor's Degree 3 - Higher Education - Degree 4 - Higher Education - Master's 5 - Higher Education - Doctorate 6 - Frequency of Higher Education 9 - 12th Year of Schooling - Not Completed 10 - 11th Year of Schooling - Not Completed 11 - 7th Year (Old) 12 - Other - 11th Year of Schooling 14 - 10th Year of Schooling 18 - General commerce course 19 - Basic Education 3rd Cycle (9th/10th/11th Year) or Equiv. 22 - Technical-professional course 26 - 7th year of schooling 27 - 2nd cycle of the general high school course 29 - 9th Year of Schooling - Not Completed 30 - 8th year of schooling 34 - Unknown 35 - Can't read or write 36 - Can read without having a 4th year of schooling 37 - Basic education 1st cycle (4th/5th year) or equiv. 38 - Basic Education 2nd Cycle (6th/7th/8th Year) or Equiv. 39 - Technological specialization course 40 - Higher education - degree (1st cycle) 41 - Specialized higher studies course 42 - Professional higher technical course 43 - Higher Education - Master (2nd cycle) 44 - Higher Education - Doctorate (3rd cycle)

In [ ]:
# analyze dropout risk by Mother's Qualification
mother_edu_risk = con.execute("""
    SELECT
        "Mother's qualification",
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY "Mother's qualification"
    ORDER BY total_students DESC
    LIMIT 10
""").df()

# Convert the ID to a string for better categorical plotting
mother_edu_risk["Mother's qualification"] = mother_edu_risk["Mother's qualification"].astype(str)

mother_edu_risk

,Mother's qualification,total_students,dropout_rate_pct,avg_sem1_grade
0,1,1069,28.06,11.02
1,22,1009,37.96,10.28
2,13,953,28.44,10.71
3,23,562,24.91,11.25
4,3,438,31.74,10.48
5,19,130,73.85,7.88
6,2,83,24.10,11.27
7,4,49,16.33,11.11
8,10,42,52.38,9.10
9,5,21,38.10,11.82


In [ ]:
# Create a bar chart for Mother's Qualification Risk
fig = px.bar(
    mother_edu_risk,
    x="Mother's qualification",
    y='dropout_rate_pct',
    title="Dropout Rate by Mother's Qualification (Top 10 Most Frequent)",
    labels={'dropout_rate_pct': 'Dropout Rate (%)', "Mother's qualification": 'Qualification ID'},
    color='dropout_rate_pct',
    color_continuous_scale='Viridis',
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(xaxis_title="Mother's Qualification ID", yaxis_title="Dropout Rate (%)", height=500)
fig.show()

**Father's Qualifications**

1 - Secondary Education - 12th Year of Schooling or Eq. 2 - Higher Education - Bachelor's Degree 3 - Higher Education - Degree 4 - Higher Education - Master's 5 - Higher Education - Doctorate 6 - Frequency of Higher Education 9 - 12th Year of Schooling - Not Completed 10 - 11th Year of Schooling - Not Completed 11 - 7th Year (Old) 12 - Other - 11th Year of Schooling 13 - 2nd year complementary high school course 14 - 10th Year of Schooling 18 - General commerce course 19 - Basic Education 3rd Cycle (9th/10th/11th Year) or Equiv. 20 - Complementary High School Course 22 - Technical-professional course 25 - Complementary High School Course - not concluded 26 - 7th year of schooling 27 - 2nd cycle of the general high school course 29 - 9th Year of Schooling - Not Completed 30 - 8th year of schooling 31 - General Course of Administration and Commerce 33 - Supplementary Accounting and Administration 34 - Unknown 35 - Can't read or write 36 - Can read without having a 4th year of schooling 37 - Basic education 1st cycle (4th/5th year) or equiv. 38 - Basic Education 2nd Cycle (6th/7th/8th Year) or Equiv. 39 - Technological specialization course 40 - Higher education - degree (1st cycle) 41 - Specialized higher studies course 42 - Professional higher technical course 43 - Higher Education - Master (2nd cycle) 44 - Higher Education - Doctorate (3rd cycle)

In [ ]:
#analyze dropout risk by Father's Qualification
father_edu_risk = con.execute("""
    SELECT
        "Father's qualification",
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY "Father's qualification"
    ORDER BY total_students DESC
    LIMIT 10
""").df()

# Convert to string for plotting
father_edu_risk["Father's qualification"] = father_edu_risk["Father's qualification"].astype(str)

father_edu_risk

,Father's qualification,total_students,dropout_rate_pct,avg_sem1_grade
0,27,1209,35.73,10.36
1,14,968,27.27,10.96
2,1,904,31.08,10.65
3,28,702,23.79,11.39
4,3,282,31.91,10.57
5,24,112,72.32,7.77
6,2,68,32.35,11.19
7,4,39,35.90,10.52
8,10,38,36.84,10.43
9,29,20,40.00,8.97


In [ ]:
# Create a bar chart for Father's Qualification Risk
fig = px.bar(
    father_edu_risk,
    x="Father's qualification",
    y='dropout_rate_pct',
    title="Dropout Rate by Father's Qualification (Top 10 Most Frequent)",
    labels={'dropout_rate_pct': 'Dropout Rate (%)', "Father's qualification": 'Qualification ID'},
    color='dropout_rate_pct',
    color_continuous_scale='Cividis',
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(xaxis_title="Father's Qualification ID", yaxis_title="Dropout Rate (%)", height=500)
fig.show()

**Parental Education**

In [ ]:
# Comparing Higher Ed vs. Non-Higher Ed for both parents

interaction_sql = """
SELECT
    CASE WHEN "Mother's qualification" IN (2, 3, 4, 5, 40, 43, 44) THEN 'Degree' ELSE 'No Degree' END as mother_level,
    CASE WHEN "Father's qualification" IN (2, 3, 4, 5, 40, 43, 44) THEN 'Degree' ELSE 'No Degree' END as father_level,
    COUNT(*) as total_students,
    ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) as dropout_rate_pct
FROM student_data
GROUP BY mother_level, father_level
"""

parental_link = con.execute(interaction_sql).df()

# Visualize as a Heatmap
fig_link = px.density_heatmap(
    parental_link,
    x="mother_level",
    y="father_level",
    z="dropout_rate_pct",
    text_auto=True,
    title="Linked Risk: Mother vs. Father Education Level",
    labels={'dropout_rate_pct': 'Dropout %', 'mother_level': "Mother's Education", 'father_level': "Father's Education"},
    color_continuous_scale='Reds'
)
fig_link.show()

**Parents Occupations**

**Mother's Occupation**

0 - Student 1 - Representatives of the Legislative Power and Executive Bodies, Directors, Directors and Executive Managers 2 - Specialists in Intellectual and Scientific Activities 3 - Intermediate Level Technicians and Professions 4 - Administrative staff 5 - Personal Services, Security and Safety Workers and Sellers 6 - Farmers and Skilled Workers in Agriculture, Fisheries and Forestry 7 - Skilled Workers in Industry, Construction and Craftsmen 8 - Installation and Machine Operators and Assembly Workers 9 - Unskilled Workers 10 - Armed Forces Professions 90 - Other Situation 99 - (blank) 122 - Health professionals 123 - teachers 125 - Specialists in information and communication technologies (ICT) 131 - Intermediate level science and engineering technicians and professions 132 - Technicians and professionals, of intermediate level of health 134 - Intermediate level technicians from legal, social, sports, cultural and similar services 141 - Office workers, secretaries in general and data processing operators 143 - Data, accounting, statistical, financial services and registry-related operators 144 - Other administrative support staff 151 - personal service workers 152 - sellers 153 - Personal care workers and the like 171 - Skilled construction workers and the like, except electricians 173 - Skilled workers in printing, precision instrument manufacturing, jewelers, artisans and the like 175 - Workers in food processing, woodworking, clothing and other industries and crafts 191 - cleaning workers 192 - Unskilled workers in agriculture, animal production, fisheries and forestry 193 - Unskilled workers in extractive industry, construction, manufacturing and transport 194 - Meal preparation assistants

In [ ]:
# analyze dropout risk by Mother's occupation
mother_occ_risk = con.execute("""
    SELECT
        "Mother's occupation",
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY "Mother's occupation"
    ORDER BY total_students DESC
    LIMIT 10
""").df()

# Convert ID to string for better categorization in the chart
mother_occ_risk["Mother's occupation"] = mother_occ_risk["Mother's occupation"].astype(str)

mother_occ_risk

,Mother's occupation,total_students,dropout_rate_pct,avg_sem1_grade
0,10,1577,31.07,10.67
1,5,817,30.35,10.89
2,6,530,29.43,10.99
3,4,351,27.07,11.03
4,3,318,32.08,10.66
5,8,272,29.41,10.91
6,1,144,68.75,8.27
7,2,102,38.24,10.19
8,7,91,28.57,10.33
9,12,70,72.86,7.54


In [ ]:
# Create a bar chart for Mother's Occupation Risk
fig = px.bar(
    mother_occ_risk,
    x="Mother's occupation",
    y='dropout_rate_pct',
    title="Dropout Rate by Mother's Occupation (Top 10 Most Frequent)",
    labels={'dropout_rate_pct': 'Dropout Rate (%)', "Mother's occupation": 'Occupation ID'},
    color='dropout_rate_pct',
    color_continuous_scale='Magma',
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(xaxis_title="Mother's Occupation ID", yaxis_title="Dropout Rate (%)", height=500)
fig.show()

**Father's Occupation**

0 - Student 1 - Representatives of the Legislative Power and Executive Bodies, Directors, Directors and Executive Managers 2 - Specialists in Intellectual and Scientific Activities 3 - Intermediate Level Technicians and Professions 4 - Administrative staff 5 - Personal Services, Security and Safety Workers and Sellers 6 - Farmers and Skilled Workers in Agriculture, Fisheries and Forestry 7 - Skilled Workers in Industry, Construction and Craftsmen 8 - Installation and Machine Operators and Assembly Workers 9 - Unskilled Workers 10 - Armed Forces Professions 90 - Other Situation 99 - (blank) 101 - Armed Forces Officers 102 - Armed Forces Sergeants 103 - Other Armed Forces personnel 112 - Directors of administrative and commercial services 114 - Hotel, catering, trade and other services directors 121 - Specialists in the physical sciences, mathematics, engineering and related techniques 122 - Health professionals 123 - teachers 124 - Specialists in finance, accounting, administrative organization, public and commercial relations 131 - Intermediate level science and engineering technicians and professions 132 - Technicians and professionals, of intermediate level of health 134 - Intermediate level technicians from legal, social, sports, cultural and similar services 135 - Information and communication technology technicians 141 - Office workers, secretaries in general and data processing operators 143 - Data, accounting, statistical, financial services and registry-related operators 144 - Other administrative support staff 151 - personal service workers 152 - sellers 153 - Personal care workers and the like 154 - Protection and security services personnel 161 - Market-oriented farmers and skilled agricultural and animal production workers 163 - Farmers, livestock keepers, fishermen, hunters and gatherers, subsistence 171 - Skilled construction workers and the like, except electricians 172 - Skilled workers in metallurgy, metalworking and similar 174 - Skilled workers in electricity and electronics 175 - Workers in food processing, woodworking, clothing and other industries and crafts 181 - Fixed plant and machine operators 182 - assembly workers 183 - Vehicle drivers and mobile equipment operators 192 - Unskilled workers in agriculture, animal production, fisheries and forestry 193 - Unskilled workers in extractive industry, construction, manufacturing and transport 194 - Meal preparation assistants 195 - Street vendors (except food) and street service providers

In [ ]:
# analyze dropout risk by Father's occupation
father_occ_risk = con.execute("""
    SELECT
        "Father's occupation",
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY "Father's occupation"
    ORDER BY total_students DESC
    LIMIT 10
""").df()

# Convert ID to string for charting
father_occ_risk["Father's occupation"] = father_occ_risk["Father's occupation"].astype(str)

father_occ_risk

,Father's occupation,total_students,dropout_rate_pct,avg_sem1_grade
0,10,1010,31.98,10.63
1,8,666,27.63,11.14
2,6,516,28.68,10.62
3,5,386,36.01,11.14
4,4,384,29.69,10.71
5,9,318,29.56,10.77
6,11,266,31.95,10.51
7,7,242,28.51,10.52
8,3,197,35.53,10.29
9,2,134,35.82,10.33


In [ ]:
# Create a bar chart for Father's Occupation Risk
fig = px.bar(
    father_occ_risk,
    x="Father's occupation",
    y='dropout_rate_pct',
    title="Dropout Rate by Father's Occupation (Top 10 Most Frequent)",
    labels={'dropout_rate_pct': 'Dropout Rate (%)', "Father's occupation": 'Occupation ID'},
    color='dropout_rate_pct',
    color_continuous_scale='Electric',
    hover_data=['total_students', 'avg_sem1_grade'],
    text='dropout_rate_pct'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(xaxis_title="Father's Occupation ID", yaxis_title="Dropout Rate (%)", height=500)
fig.show()

**Parental Occupation**

In [ ]:
# Professional/Admin vs. Other occupations for both parents
occ_interaction_sql = """
SELECT
    CASE WHEN "Mother's occupation" IN (1, 2, 3, 4, 122, 123, 124, 125, 131, 132, 134, 135) THEN 'Professional/Admin' ELSE 'Other/Manual' END as mother_occ_level,
    CASE WHEN "Father's occupation" IN (1, 2, 3, 4, 122, 123, 124, 125, 131, 132, 134, 135) THEN 'Professional/Admin' ELSE 'Other/Manual' END as father_occ_level,
    COUNT(*) as total_students,
    ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) as dropout_rate_pct
FROM student_data
GROUP BY mother_occ_level, father_occ_level
"""

parental_occ_link = con.execute(occ_interaction_sql).df()

parental_occ_link

,mother_occ_level,father_occ_level,total_students,dropout_rate_pct
0,Professional/Admin,Professional/Admin,475,40.63
1,Other/Manual,Professional/Admin,368,32.88
2,Other/Manual,Other/Manual,3141,30.72
3,Professional/Admin,Other/Manual,440,32.27


In [ ]:
# Create the heatmap for Parental Occupation Interaction
fig_occ = px.imshow(
    parental_occ_link.pivot(index='father_occ_level', columns='mother_occ_level', values='dropout_rate_pct'),
    labels=dict(x="Mother's Occupation", y="Father's Occupation", color="Dropout Rate (%)"),
    x=['Other/Manual', 'Professional/Admin'],
    y=['Other/Manual', 'Professional/Admin'],
    color_continuous_scale='Reds',
    title='Linked Risk Heatmap: Mother vs. Father Occupation Level',
    text_auto='.1f'
)

fig_occ.update_layout(height=500)
fig_occ.show()

**Parents Qualifications and Grades**

In [ ]:
#average grades by Mother's Qualification
mother_grade_analysis = con.execute("""
    SELECT
        "Mother's qualification",
        COUNT(*) AS total_students,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade,
        ROUND(AVG("Curricular units 2nd sem (grade)"), 2) AS avg_sem2_grade
    FROM student_data
    GROUP BY "Mother's qualification"
    ORDER BY total_students DESC
    LIMIT 10
""").df()

# Convert ID to string for plotting
mother_grade_analysis["Mother's qualification"] = mother_grade_analysis["Mother's qualification"].astype(str)

mother_grade_analysis

,Mother's qualification,total_students,avg_sem1_grade,avg_sem2_grade
0,1,1069,11.02,10.62
1,22,1009,10.28,9.95
2,13,953,10.71,10.32
3,23,562,11.25,10.92
4,3,438,10.48,10.09
5,19,130,7.88,6.55
6,2,83,11.27,10.62
7,4,49,11.11,10.79
8,10,42,9.10,8.62
9,5,21,11.82,11.26


In [ ]:
# Create a comparison chart for Mother's Qualification impact on Grades
fig = px.bar(
    mother_grade_analysis,
    x="Mother's qualification",
    y='avg_sem1_grade',
    title="Average 1st Sem Grade by Mother's Qualification",
    labels={'avg_sem1_grade': 'Average Grade', "Mother's qualification": 'Qualification ID'},
    color='avg_sem1_grade',
    color_continuous_scale='Blues',
    text='avg_sem1_grade'
)

fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(xaxis_title="Mother's Qualification ID", yaxis_range=[0, 20], height=500)
fig.show()

In [ ]:
#average grades by Father's Qualification
father_grade_analysis = con.execute("""
    SELECT
        "Father's qualification",
        COUNT(*) AS total_students,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade,
        ROUND(AVG("Curricular units 2nd sem (grade)"), 2) AS avg_sem2_grade
    FROM student_data
    GROUP BY "Father's qualification"
    ORDER BY total_students DESC
    LIMIT 10
""").df()

# Convert ID to string for plotting
father_grade_analysis["Father's qualification"] = father_grade_analysis["Father's qualification"].astype(str)

father_grade_analysis

,Father's qualification,total_students,avg_sem1_grade,avg_sem2_grade
0,27,1209,10.36,9.99
1,14,968,10.96,10.62
2,1,904,10.65,10.28
3,28,702,11.39,11.00
4,3,282,10.57,10.11
5,24,112,7.77,6.75
6,2,68,11.19,10.24
7,4,39,10.52,10.54
8,10,38,10.43,10.18
9,29,20,8.97,7.58


In [ ]:
# Create a comparison chart for Father's Qualification impact on Grades
fig = px.bar(
    father_grade_analysis,
    x="Father's qualification",
    y='avg_sem1_grade',
    title="Average 1st Sem Grade by Father's Qualification",
    labels={'avg_sem1_grade': 'Average Grade', "Father's qualification": 'Qualification ID'},
    color='avg_sem1_grade',
    color_continuous_scale='Cividis',
    text='avg_sem1_grade'
)

# Improve formatting to match the previous chart
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(
    xaxis_title="Father's Qualification ID",
    yaxis_title="Average Grade",
    yaxis_range=[0, 20],
    height=500
)

fig.show()

In [ ]:
# Comparing Higher Ed vs. Non-Higher Ed impact on Grades

grade_interaction_sql = """
SELECT
    CASE WHEN "Mother's qualification" IN (2, 3, 4, 5, 40, 43, 44) THEN 'Higher Ed' ELSE 'Non-Higher Ed' END as mother_edu,
    CASE WHEN "Father's qualification" IN (2, 3, 4, 5, 40, 43, 44) THEN 'Higher Ed' ELSE 'Non-Higher Ed' END as father_edu,
    COUNT(*) as total_students,
    ROUND(AVG("Curricular units 1st sem (grade)"), 2) as avg_grade
FROM student_data
GROUP BY mother_edu, father_edu
"""

grade_link = con.execute(grade_interaction_sql).df()
grade_link

,mother_edu,father_edu,total_students,avg_grade
0,Non-Higher Ed,Higher Ed,175,10.29
1,Higher Ed,Higher Ed,232,10.91
2,Non-Higher Ed,Non-Higher Ed,3658,10.65
3,Higher Ed,Non-Higher Ed,359,10.55


In [ ]:
# Create the heatmap for Parental Education interaction on Grades
fig_grade_link = px.imshow(
    grade_link.pivot(index='father_edu', columns='mother_edu', values='avg_grade'),
    labels=dict(x="Mother's Education", y="Father's Education", color="Avg Grade"),
    x=['Higher Ed', 'Non-Higher Ed'],
    y=['Higher Ed', 'Non-Higher Ed'],
    color_continuous_scale='Blues',
    title='Linked Academic Performance: Mother vs. Father Education',
    text_auto='.2f'
)

fig_grade_link.update_layout(height=500)
fig_grade_link.show()

# **Accademic Factors**

**Semester Comparison**

In [ ]:
#compare units enrolled vs approved across both semesters
academic_comparison = con.execute("""
    SELECT
        Target,
        ROUND(AVG("Curricular units 1st sem (enrolled)"), 2) AS sem1_enrolled,
        ROUND(AVG("Curricular units 1st sem (approved)"), 2) AS sem1_approved,
        ROUND(AVG("Curricular units 2nd sem (enrolled)"), 2) AS sem2_enrolled,
        ROUND(AVG("Curricular units 2nd sem (approved)"), 2) AS sem2_approved
    FROM student_data
    GROUP BY Target
""").df()

academic_comparison

,Target,sem1_enrolled,sem1_approved,sem2_enrolled,sem2_approved
0,Dropout,5.82,2.55,5.78,1.94
1,Graduate,6.67,6.23,6.63,6.18
2,Enrolled,5.96,4.32,5.94,4.06


In [ ]:
# Reshape the data for a better grouped bar chart
melted_academic = academic_comparison.melt(id_vars='Target', var_name='Metric', value_name='Average Units')

fig = px.bar(
    melted_academic,
    x='Target',
    y='Average Units',
    color='Metric',
    barmode='group',
    title='Academic Progress: Enrolled vs. Approved Units by Semester',
    labels={'Average Units': 'Avg Curricular Units'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig.update_layout(height=500)
fig.show()

**Enrolled vs. Evaluated**

In [ ]:
#units enrolled vs evaluated across both semesters
eval_comparison = con.execute("""
    SELECT
        Target,
        ROUND(AVG("Curricular units 1st sem (enrolled)"), 2) AS sem1_enrolled,
        ROUND(AVG("Curricular units 1st sem (evaluations)"), 2) AS sem1_evaluations,
        ROUND(AVG("Curricular units 2nd sem (enrolled)"), 2) AS sem2_enrolled,
        ROUND(AVG("Curricular units 2nd sem (evaluations)"), 2) AS sem2_evaluations
    FROM student_data
    GROUP BY Target
""").df()

eval_comparison

,Target,sem1_enrolled,sem1_evaluations,sem2_enrolled,sem2_evaluations
0,Dropout,5.82,7.75,5.78,7.17
1,Graduate,6.67,8.28,6.63,8.14
2,Enrolled,5.96,9.34,5.94,9.44


In [ ]:
# Reshape the data for a grouped bar chart
melted_eval = eval_comparison.melt(id_vars='Target', var_name='Metric', value_name='Average Units')

fig = px.bar(
    melted_eval,
    x='Target',
    y='Average Units',
    color='Metric',
    barmode='group',
    title='Academic Effort: Enrolled vs. Evaluated Units by Semester',
    labels={'Average Units': 'Avg Curricular Units'},
    color_discrete_sequence=px.colors.qualitative.Prism
)

fig.update_layout(height=500)
fig.show()

**Success Rate**

In [ ]:
# calculate the Success Rate (Approved / Evaluated)
success_rate_analysis = con.execute("""
    SELECT
        Target,
        ROUND(AVG("Curricular units 1st sem (approved)") / NULLIF(AVG("Curricular units 1st sem (evaluations)"), 0) * 100, 2) AS sem1_success_rate,
        ROUND(AVG("Curricular units 2nd sem (approved)") / NULLIF(AVG("Curricular units 2nd sem (evaluations)"), 0) * 100, 2) AS sem2_success_rate
    FROM student_data
    GROUP BY Target
""").df()

success_rate_analysis

,Target,sem1_success_rate,sem2_success_rate
0,Dropout,32.92,27.05
1,Graduate,75.30,75.86
2,Enrolled,46.23,43.01


In [ ]:
# Reshape the data for a grouped bar chart
melted_success = success_rate_analysis.melt(id_vars='Target', var_name='Semester', value_name='Success Rate (%)')

# Create a grouped bar chart for Success Rate
fig = px.bar(
    melted_success,
    x='Target',
    y='Success Rate (%)',
    color='Semester',
    barmode='group',
    title='Passing Efficiency: Success Rate (Approved / Evaluated) by Semester',
    labels={'Success Rate (%)': 'Passing Rate (%)', 'Target': 'Student Status'},
    color_discrete_map={'sem1_success_rate': 'lightseagreen', 'sem2_success_rate': 'darkcyan'},
    text='Success Rate (%)'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 110], height=500)
fig.show()

**Grades**

In [ ]:
# compare average grades across both semesters
grade_comparison = con.execute("""
    SELECT
        Target,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade,
        ROUND(AVG("Curricular units 2nd sem (grade)"), 2) AS avg_sem2_grade
    FROM student_data
    GROUP BY Target
""").df()

grade_comparison

,Target,avg_sem1_grade,avg_sem2_grade
0,Dropout,7.26,5.90
1,Graduate,12.64,12.70
2,Enrolled,11.13,11.12


In [ ]:
# Reshape the data for a grouped bar chart
melted_grades = grade_comparison.melt(id_vars='Target', var_name='Semester', value_name='Average Grade')

# Create a grouped bar chart for Grade Comparison
fig = px.bar(
    melted_grades,
    x='Target',
    y='Average Grade',
    color='Semester',
    barmode='group',
    title='Academic Performance: 1st vs. 2nd Semester Grades',
    labels={'Average Grade': 'Avg Grade', 'Target': 'Student Status'},
    color_discrete_map={'avg_sem1_grade': 'royalblue', 'avg_sem2_grade': 'cornflowerblue'},
    text='Average Grade'
)

fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(yaxis_range=[0, 16], height=500)
fig.show()

**Previous Qualifications and Grades**

1 - Secondary education 2 - Higher education - bachelor's degree 3 - Higher education - degree 4 - Higher education - master's 5 - Higher education - doctorate 6 - Frequency of higher education 9 - 12th year of schooling - not completed 10 - 11th year of schooling - not completed 12 - Other - 11th year of schooling 14 - 10th year of schooling 15 - 10th year of schooling - not completed 19 - Basic education 3rd cycle (9th/10th/11th year) or equiv. 38 - Basic education 2nd cycle (6th/7th/8th year) or equiv. 39 - Technological specialization course 40 - Higher education - degree (1st cycle) 42 - Professional higher technical course 43 - Higher education - master (2nd cycle)

In [ ]:
# grades based on previous educational background
prev_qual_grade_analysis = con.execute("""
    SELECT
        "Previous qualification",
        COUNT(*) AS total_students,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade,
        ROUND(AVG("Curricular units 2nd sem (grade)"), 2) AS avg_sem2_grade
    FROM student_data
    GROUP BY "Previous qualification"
    ORDER BY total_students DESC
    LIMIT 10
""").df()

# Convert ID to string for better plotting
prev_qual_grade_analysis["Previous qualification"] = prev_qual_grade_analysis["Previous qualification"].astype(str)

prev_qual_grade_analysis

,Previous qualification,total_students,avg_sem1_grade,avg_sem2_grade
0,1,3717,10.80,10.42
1,14,219,11.21,11.04
2,12,162,8.55,7.22
3,3,126,9.19,8.22
4,9,45,6.85,6.80
5,15,40,11.95,12.17
6,16,36,12.62,12.48
7,2,23,8.64,8.36
8,6,16,11.81,10.82
9,7,11,1.18,2.23


In [ ]:
# Create a bar chart for Previous Qualification impact on Grades
fig = px.bar(
    prev_qual_grade_analysis,
    x="Previous qualification",
    y='avg_sem1_grade',
    title="Average 1st Sem Grade by Previous Qualification (Top 10)",
    labels={'avg_sem1_grade': 'Average Grade', "Previous qualification": 'Qualification ID'},
    color='avg_sem1_grade',
    color_continuous_scale='Greens',
    text='avg_sem1_grade'
)

fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(xaxis_title="Previous Qualification ID", yaxis_range=[0, 18], height=500)
fig.show()

**Killer Courses**

33 - Biofuel Production Technologies 171 - Animation and Multimedia Design 8014 - Social Service (evening attendance) 9003 - Agronomy 9070 - Communication Design 9085 - Veterinary Nursing 9119 - Informatics Engineering 9130 - Equinculture 9147 - Management 9238 - Social Service 9254 - Tourism 9500 - Nursing 9556 - Oral Hygiene 9670 - Advertising and Marketing Management 9773 - Journalism and Communication 9853 - Basic Education 9991 - Management (evening attendance)

In [ ]:
# analyze dropout risk and grades by Course
course_analysis = con.execute("""
    SELECT
        Course,
        COUNT(*) AS total_students,
        ROUND(AVG(CASE WHEN Target = 'Dropout' THEN 1 ELSE 0 END) * 100, 2) AS dropout_rate_pct,
        ROUND(AVG("Curricular units 1st sem (grade)"), 2) AS avg_sem1_grade
    FROM student_data
    GROUP BY Course
    ORDER BY total_students DESC
""").df()

# Convert Course ID to string for better categorization in the chart
course_analysis['Course'] = course_analysis['Course'].astype(str)

course_analysis

,Course,total_students,dropout_rate_pct,avg_sem1_grade
0,12,766,15.40,12.46
1,9,380,35.26,10.12
2,10,355,18.31,11.24
3,6,337,26.71,11.86
4,15,331,30.51,11.69
5,17,268,50.75,9.10
6,14,268,35.45,11.33
7,11,252,38.10,10.46
8,5,226,22.57,12.08
9,2,215,38.14,2.07


In [ ]:
# Dictionary mapping Course IDs (1-17) to names
course_labels = {
    1: 'Biofuel Production Technologies',
    2: 'Animation and Multimedia Design',
    3: 'Social Service (evening attendance)',
    4: 'Agronomy',
    5: 'Communication Design',
    6: 'Veterinary Nursing',
    7: 'Informatics Engineering',
    8: 'Equinculture',
    9: 'Management',
    10: 'Social Service',
    11: 'Tourism',
    12: 'Nursing',
    13: 'Oral Hygiene',
    14: 'Advertising and Marketing Management',
    15: 'Journalism and Communication',
    16: 'Basic Education',
    17: 'Management (evening attendance)'
}

# Apply the labels to the course_analysis DataFrame
course_analysis['Course_Name'] = course_analysis['Course'].astype(int).map(course_labels)

# Display the updated table
course_analysis[['Course', 'Course_Name', 'total_students', 'dropout_rate_pct', 'avg_sem1_grade']]

,Course,Course_Name,total_students,dropout_rate_pct,avg_sem1_grade
0,12,Nursing,766,15.40,12.46
1,9,Management,380,35.26,10.12
2,10,Social Service,355,18.31,11.24
3,6,Veterinary Nursing,337,26.71,11.86
4,15,Journalism and Communication,331,30.51,11.69
5,17,Management (evening attendance),268,50.75,9.10
6,14,Advertising and Marketing Management,268,35.45,11.33
7,11,Tourism,252,38.10,10.46
8,5,Communication Design,226,22.57,12.08
9,2,Animation and Multimedia Design,215,38.14,2.07


In [ ]:
# Create the labeled scatter plot
fig = px.scatter(
    course_analysis,
    x='avg_sem1_grade',
    y='dropout_rate_pct',
    size='total_students',
    color='Course_Name',
    hover_name='Course_Name',
    title='Course Performance: Grade vs. Dropout Rate',
    labels={'avg_sem1_grade': 'Avg 1st Sem Grade', 'dropout_rate_pct': 'Dropout Rate (%)', 'Course_Name': 'Course'},
    text='Course_Name'
)

# Move labels above the points and clean up layout
fig.update_traces(textposition='top center')

# Update layout to show the legend and adjust dimensions
fig.update_layout(
    height=700,
    showlegend=True,
    legend_title_text='Course Name',
    margin=dict(l=50, r=50, t=100, b=100)
)

fig.show()

In [ ]:
!pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 66.5 MB/s eta 0:00:00
